# 01 – Train ACT on ALOHA TransferCube (baseline)

Trains an [ACT](https://arxiv.org/abs/2304.13705) policy from scratch with LeRobot on the
`lerobot/aloha_sim_transfer_cube_human` dataset (50 human demonstrations, MuJoCo ALOHA sim),
then evaluates it on 50 simulated episodes.

**Kaggle setup:** Settings → Accelerator: *GPU T4 x2* (or P100), Internet: *On*.
Run in the background with **Save Version → Save & Run All (Commit)**; everything written to
`/kaggle/working` is kept as notebook output.

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
REPO_URL = "https://github.com/mrtelisa/lerobot_force_insertion.git"
!git clone --depth 1 $REPO_URL /kaggle/working/repo
!pip install -q -r /kaggle/working/repo/requirements.txt
!lerobot-info

fatal: could not create leading directories of '/kaggle/working/repo': Permission denied
ERROR: Could not open requirements file: [Errno 2] No such file or directory: '/kaggle/working/repo/requirements.txt'
- LeRobot version: 0.7.0
- Platform: Linux-7.0.0-38-generic-x86_64-with-glibc2.39
- Python version: 3.12.3
- Huggingface Hub version: 2.4.0
- Transformers version: N/A
- Datasets version: 5.2.0
- Numpy version: 2.2.6
- FFmpeg version: 6.1.1-3ubuntu5
- PyTorch version: 2.15.0+cpu
- Torchcodec version: 0.17.0 (import failed: OSError)
- Is PyTorch built with CUDA support?: False
- Cuda version: N/A
- GPU model: N/A
- Using GPU in script?: <fill in>
- lerobot scripts: ['lerobot-annotate', 'lerobot-build-mp4-sidecar', 'lerobot-calibrate', 'lerobot-convert-dcp', 'lerobot-dataset-viz', 'lerobot-edit-dataset', 'lerobot-env-server', 'lerobot-eval', 'lerobot-find-cameras', 'lerobot-find-joint-limits', 'lerobot-find-port', 'lerobot-imgtransform-viz', 'lerobot-info', 'lerobot-policy-server', 'l

In [ ]:
import torch
print(torch.__version__, "CUDA build:", torch.version.cuda, "available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU visible: check the accelerator setting or the torch CUDA build vs the Kaggle driver"

## Training

Defaults of LeRobot 0.7.0 for ACT: `batch_size=8`, chunk size 100, ResNet-18 backbone.
No `--env.type` is passed, so there is no simulation rollout during training (evaluation is done separately below).
Check the `step` / `updt_s` values in the log during the first minutes to estimate the total time;
lower `STEPS` if 100k steps do not fit in the 12 h Kaggle session limit.

In [ ]:
STEPS = 100_000
OUT = "/kaggle/working/outputs/train/act_transfer_cube_human"

In [ ]:
!lerobot-train \
  --policy.type=act \
  --dataset.repo_id=lerobot/aloha_sim_transfer_cube_human \
  --output_dir=$OUT \
  --job_name=act_transfer_cube_human \
  --policy.device=cuda \
  --policy.push_to_hub=false \
  --steps=$STEPS \
  --batch_size=8 \
  --save_freq=20000 \
  --log_freq=200 \
  --num_workers=4 \
  --seed=1000

## Evaluation (50 episodes)

Headless rendering on Kaggle needs `MUJOCO_GL=egl`. If EGL fails, try `osmesa`, or download the
checkpoint and run the evaluation locally (about 16 s per episode on CPU).

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
CKPT = f"{OUT}/checkpoints/last/pretrained_model"
EVAL_OUT = "/kaggle/working/outputs/eval/act_transfer_cube_human"

In [ ]:
!lerobot-eval \
  --policy.path=$CKPT \
  --env.type=aloha \
  --env.task=AlohaTransferCube-v0 \
  --eval.n_episodes=50 \
  --eval.batch_size=10 \
  --policy.device=cuda \
  --policy.use_amp=false \
  --output_dir=$EVAL_OUT

In [ ]:
import json
overall = json.load(open(f"{EVAL_OUT}/eval_info.json"))["overall"]
print({k: v for k, v in overall.items() if "video" not in k})

## Keep the output small

Copy the final model weights (no optimizer state) to a plain folder, so the downloaded zip does not depend on the `last` symlink, then delete the intermediate checkpoints.


In [ ]:
!rm -rf /kaggle/working/repo
!cp -rL $OUT/checkpoints/last/pretrained_model /kaggle/working/outputs/final_model
!rm -rf $OUT/checkpoints
!ls /kaggle/working/outputs/final_model
!du -sh /kaggle/working/outputs